# LeetCode #1350: Product of the Last K Numbers

https://leetcode.com/problems/product-of-the-last-k-numbers/

## Comparison of Approaches

| Approach | Time Complexity | Space Complexity |
| :--- | :--- | :--- |
| **Brute Force** | $O(k)$ per query | $O(n)$ |
| **Optimal: Prefix Product with Zero Reset ★** | $O(1)$ per query | $O(n)$ |

---

## Understanding the Methods

### Brute Force
Keep all added numbers in a list. For `getProduct(k)`, multiply the last $k$ elements. $O(k)$ per query, acceptable but unnecessarily slow when queries are frequent.

### Optimal: Prefix Product with Zero Reset ★
Maintain a running prefix-product array. `add(num)` appends `prefix[-1] * num`. `getProduct(k)` divides the last prefix by the prefix $k$ steps back — constant time. When a zero is added, reset the prefix array to `[1]` because zero nullifies all earlier products; if `k` spans into the reset boundary, return 0 immediately.

**Why this is better than Brute Force:** Division replaces a $k$-step multiply loop, turning every query into $O(1)$ regardless of $k$.

**Constraints:**
* $0 \leq \text{num} \leq 10^9$
* $1 \leq k \leq 40000$
* At most $40000$ calls in total to `add` and `getProduct`
* The product of the last `k` numbers fits in a 32-bit integer


## Solutions

### C#

In [ ]:
public class ProductOfNumbers {
    private List<long> prefix = new() { 1 };

    public void Add(int num) {
        if (num == 0)
            // A zero wipes out all past products — restart the prefix chain
            prefix = new List<long> { 1 };
        else
            prefix.Add(prefix[^1] * num);
    }

    public int GetProduct(int k) {
        // If k reaches back past the last zero reset, the window contains a zero
        if (k >= prefix.Count) return 0;
        // Divide out the prefix before the window to isolate the last k products
        return (int)(prefix[^1] / prefix[prefix.Count - 1 - k]);
    }
}

### Python

In [ ]:
class ProductOfNumbers:
    def __init__(self):
        self.prefix = [1]

    def add(self, num: int) -> None:
        if num == 0:
            # A zero wipes out all past products — restart the prefix chain
            self.prefix = [1]
        else:
            self.prefix.append(self.prefix[-1] * num)

    def get_product(self, k: int) -> int:
        # If k reaches back past the last zero reset, the window contains a zero
        if k >= len(self.prefix):
            return 0
        # Divide out the prefix before the window to isolate the last k products
        return self.prefix[-1] // self.prefix[-1 - k]

### Go

In [ ]:
type ProductOfNumbers struct {
    prefix []int
}

func Constructor() ProductOfNumbers {
    return ProductOfNumbers{prefix: []int{1}}
}

func (p *ProductOfNumbers) Add(num int) {
    if num == 0 {
        // A zero wipes out all past products — restart the prefix chain
        p.prefix = []int{1}
    } else {
        p.prefix = append(p.prefix, p.prefix[len(p.prefix)-1]*num)
    }
}

func (p *ProductOfNumbers) GetProduct(k int) int {
    // If k reaches back past the last zero reset, the window contains a zero
    if k >= len(p.prefix) {
        return 0
    }
    n := len(p.prefix)
    // Divide out the prefix before the window to isolate the last k products
    return p.prefix[n-1] / p.prefix[n-1-k]
}

### Rust

In [ ]:
struct ProductOfNumbers {
    prefix: Vec<i64>,
}

impl ProductOfNumbers {
    fn new() -> Self {
        ProductOfNumbers { prefix: vec![1] }
    }

    fn add(&mut self, num: i32) {
        if num == 0 {
            // A zero wipes out all past products — restart the prefix chain
            self.prefix = vec![1];
        } else {
            let last = *self.prefix.last().unwrap();
            self.prefix.push(last * num as i64);
        }
    }

    fn get_product(&self, k: i32) -> i32 {
        let k = k as usize;
        let n = self.prefix.len();
        // If k reaches back past the last zero reset, the window contains a zero
        if k >= n { return 0; }
        // Divide out the prefix before the window to isolate the last k products
        (self.prefix[n - 1] / self.prefix[n - 1 - k]) as i32
    }
}

## Example Scenarios

### 1. Common Case
**Input:** `add(3)`, `add(7)`, `add(2)`, `getProduct(2)`
Prefix: `[1,3,21,42]`. `getProduct(2)`: `42 / 3 = 14` (product of last 2: $7 \times 2$). Single division yields the answer in $O(1)$.

### 2. Slightly Complex
**Input:** `add(3)`, `add(0)`, `add(2)`, `getProduct(2)`
After the zero, prefix resets to `[1]`, then becomes `[1,2]`. `getProduct(2)`: $k=2 \geq$ len(prefix)$=2$ — window spans the reset boundary, return **0** immediately.

### 3. Edge Case: Time Factor
**Input:** 40,000 `add` calls followed by 40,000 `getProduct(k)` calls
Each `add` is $O(1)$; each `getProduct` is $O(1)$. Total $O(n)$ vs brute force $O(n \cdot k)$ — the prefix approach stays linear even under maximum load.

### 4. Edge Case: Space Factor
**Input:** 40,000 non-zero `add` calls (no resets)
Prefix array grows to length 40,001. No resets ever free memory. This is the worst-case $O(n)$ space scenario — every product is preserved.

### 5. Almost-Impossible but Plausible
**Input:** Alternating zeros and non-zeros, then `getProduct(1)` after the last non-zero
Each zero triggers a reset; the prefix stays tiny (length ≤ 2). `getProduct(1)` queries the single most-recent value — always valid. Confirms that frequent resets do not break the invariant.
